# 04 — Churn Modeling

Trains the XGBoost churn classifier (`src/churn_model.py`) on the full
feature-engineered, segmented dataset. Per CLAUDE.md Section 6:
- `scale_pos_weight` recomputed per fold (not globally) to avoid leakage
- Stratified 5-fold CV, since positive class is ~26.5%
- Reporting Recall / ROC-AUC / PR-AUC, not raw accuracy
- Excludes `customerID`, `Churn` (target), `future_ltv_12m` (wrong-model
  target), `TotalCharges` (leakage risk vs. tenure x MonthlyCharges)

Prerequisite: run 01–03 first, or rebuild the pipeline inline below.

## Setup

In [1]:
import sys
sys.path.insert(0, "..")
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s")

import pandas as pd
import numpy as np

from src.data_loader import load_raw_data
from src.preprocessing import clean_data
from src.feature_engineering import engineer_features
from src.segmentation import segment_customers
from src.churn_model import (
    get_feature_columns,
    cross_validate_churn_model,
    train_final_model,
    save_model,
    predict_churn_probability,
    flag_high_risk,
)

## Rebuild the pipeline input

Loads raw data through segmentation, so this notebook is runnable
standalone. `k=5` is the value decided in `03_segmentation.ipynb` after
comparing against k=6 (near-tied silhouette) — **do not leave `k=None`
here**, that re-runs the full elbow/silhouette search every time.

In [2]:
df = load_raw_data()
df = clean_data(df)
df = engineer_features(df, random_state=42)
df, cluster_profiles = segment_customers(df, k=5, random_state=42)

print("Pipeline input shape:", df.shape)
print("Churn rate:", (df["Churn"] == "Yes").mean().round(4))

Synthetic feature sanity check (mean by Churn):
       days_since_last_activity  days_active_last_90d  \
Churn                                                   
No                        10.63                 38.51   
Yes                       15.57                 22.12   

       avg_session_duration_min  weekend_activity_ratio  
Churn                                                    
No                        14.49                    0.28  
Yes                       10.13                    0.30  


Pipeline input shape: (7043, 37)
Churn rate: 0.2654


## Feature set

Sanity-check which columns actually enter the model before training —
confirms the exclusion list (`customerID`, `Churn`, `future_ltv_12m`,
`future_ltv_12m_tier`, `TotalCharges`) is being applied, and that
segmentation output (`Cluster_ID`, `Segment_Name`) is included.

In [3]:
feature_cols = get_feature_columns(df)
print(f"{len(feature_cols)} features:")
feature_cols

32 features:


['Cluster_ID',
 'Contract',
 'Dependents',
 'DeviceProtection',
 'F_score',
 'InternetService',
 'M_score',
 'MonthlyCharges',
 'MultipleLines',
 'OnlineBackup',
 'OnlineSecurity',
 'PaperlessBilling',
 'Partner',
 'PaymentMethod',
 'PhoneService',
 'RFM_Score',
 'RFM_Segment',
 'R_score',
 'Segment_Name',
 'SeniorCitizen',
 'StreamingMovies',
 'StreamingTV',
 'TechSupport',
 'avg_addon_spend',
 'avg_session_duration_min',
 'days_active_last_90d',
 'days_since_last_activity',
 'gender',
 'monthly_spend_trend_pct',
 'service_usage_interval_days',
 'tenure',
 'weekend_activity_ratio']

## Cross-validation

Stratified 5-fold. `scale_pos_weight` is logged per fold — expect it to
hover near the dataset's overall negative:positive ratio (~2.77) each time,
confirming it's genuinely being recomputed from each fold's training rows
and not reused from a single global calculation.

In [4]:
cv_results = cross_validate_churn_model(df, n_splits=5, random_state=42)
cv_results

Prepared feature matrix: 7043 rows, 32 features (18 categorical), positive class rate=0.265


Evaluation @ threshold=0.50: recall=0.743, roc_auc=0.837, pr_auc=0.629


Fold 1/5 complete (scale_pos_weight=2.769)


Evaluation @ threshold=0.50: recall=0.746, roc_auc=0.833, pr_auc=0.637


Fold 2/5 complete (scale_pos_weight=2.769)


...

Fold 4/5 complete (scale_pos_weight=2.767)


Evaluation @ threshold=0.50: recall=0.743, roc_auc=0.829, pr_auc=0.633


Fold 5/5 complete (scale_pos_weight=2.769)


CV complete (5 folds). Mean recall=0.727, roc_auc=0.829, pr_auc=0.626


,fold,scale_pos_weight,recall,roc_auc,pr_auc
0,1,2.768562,0.743316,0.836971,0.629181
1,2,2.768562,0.745989,0.833444,0.636713
2,3,2.768562,0.729947,0.835906,0.631117
3,4,2.766711,0.670241,0.809276,0.597991
4,5,2.769231,0.743316,0.828851,0.632631


In [5]:
print("Mean CV metrics:")
cv_results[["recall", "roc_auc", "pr_auc"]].mean().round(4)

Mean CV metrics:


recall     0.7266
roc_auc    0.8289
pr_auc     0.6255
dtype: float64

## Final model (train/test split)

This is the model that gets saved and consumed downstream by
`explainability.py`. `X_test` / `y_test` are returned here so SHAP can
explain genuinely held-out rows, not training rows.

In [6]:
model, X_train, X_test, y_train, y_test, test_metrics = train_final_model(
    df, test_size=0.2, random_state=42
)
test_metrics

Prepared feature matrix: 7043 rows, 32 features (18 categorical), positive class rate=0.265


Evaluation @ threshold=0.50: recall=0.725, roc_auc=0.826, pr_auc=0.632


Final model test-set confusion matrix:
[[792 243]
 [103 271]]


Final model test-set classification report:
              precision    recall  f1-score   support

    No Churn       0.88      0.77      0.82      1035
       Churn       0.53      0.72      0.61       374

    accuracy                           0.75      1409
   macro avg       0.71      0.74      0.72      1409
weighted avg       0.79      0.75      0.76      1409



{'recall': 0.7245989304812834,
 'roc_auc': 0.8263814616755794,
 'pr_auc': 0.632007112435973}

## Feature importance (quick sanity check, not a substitute for SHAP in 06)

In [7]:
importances = model.feature_importances_
feat_names = X_train.columns.tolist()
importance_df = (
    pd.DataFrame({"feature": feat_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
importance_df

,feature,importance
0,Contract,0.205560
1,InternetService,0.130536
2,M_score,0.062274
3,RFM_Score,0.051463
4,OnlineSecurity,0.045097
5,TechSupport,0.036336
6,F_score,0.034372
7,Cluster_ID,0.028152
8,days_active_last_90d,0.027056
9,tenure,0.025314


## Save the model

Writes to `models/churn_model.joblib` (via `config.MODELS_DIR`).

In [8]:
save_model(model)

Saved churn model to /home/claude/project/models/churn_model.joblib


PosixPath('/home/claude/project/models/churn_model.joblib')

## Note on calibration (CLAUDE.md Section 6)

`scale_pos_weight` improves recall on the minority class but skews
predicted probabilities away from true calibrated probabilities.
Predicted churn probabilities below are for **ranking/prioritization**
at the 0.5 threshold, not literal probabilities — this is a documented
v1 caveat, not a bug.

In [9]:
from src.churn_model import prepare_features

X_full, _ = prepare_features(df)
df["churn_probability"] = predict_churn_probability(model, X_full)
df["churn_risk_flag"] = flag_high_risk(df["churn_probability"].to_numpy())

print("High churn risk customers:", int(df["churn_risk_flag"].sum()),
      f"({df['churn_risk_flag'].mean():.1%} of base)")
df[["customerID", "churn_probability", "churn_risk_flag"]].head()

Prepared feature matrix: 7043 rows, 32 features (18 categorical), positive class rate=0.265


High churn risk customers: 2546 (36.1% of base)


,customerID,churn_probability,churn_risk_flag
0,7590-VHVEG,0.825703,True
1,5575-GNVDE,0.093387,False
2,3668-QPYBK,0.753099,True
3,7795-CFOCW,0.048287,False
4,9237-HQITU,0.850077,True
